# Split Data into Training, Validation and Test Sets


### Initialization

In [1]:
%load_ext autoreload
%autoreload 2
from config.settings_data import DataSetsSettings
from utils.templates_split import (
    ClinicalCondition,
    DetectorType,
    SplitType,
    View
)

profile = "default"
settings = DataSetsSettings.build(profile)

## AP-Detection Task 

 * Clinical condition: Healthy
 * patient views: PA
 * Aquisition parameters: D110, S120
 * mAs range: (0,6.9]


In [ ]:
from utils.filtering_utils import filter_images, filter_patients, clear_patients, split_datasets
from utils.image_preprocessing import dcm_to_png_conversion

d110_PA_healthy = filter_images(
    settings.DB_name,
    110,
    DetectorType.DIRECT,
    (0,6.9),
    ClinicalCondition.HEALTHY,
    View.PA,
)

s120_PA_healthy = filter_images(
    settings.DB_name,
    120,
    DetectorType.SCINTILLATOR,
    (0,6.9),
    ClinicalCondition.HEALTHY,
    View.PA,
)

intersecting_pids = filter_patients(
    d110_PA_healthy, 
    s120_PA_healthy
)

clear_patients(
    d110_PA_healthy,
    s120_PA_healthy,
    intersecting_pids
)

split_report, error_report = split_datasets(
    dictionary_a=d110_PA_healthy,
    dictionary_b=s120_PA_healthy,
    split_type=SplitType.BALANCED,
    dataset_name="d110_s120_PA_acquisition_parameters_detection",
    config_path=settings.ap_detection_config,
    data_path=settings.ap_detection_data,
    version=settings.ap_detection_version,
    image_converting_function=dcm_to_png_conversion,
    random_state=42,
)

#### Dataset Pasport Generation

In [3]:
from utils.experiments_metadata import get_AP_detection_dataset_metadata
from utils.database_reports import generate_binary_dataset_pdf_report
from pathlib import Path

metadata = get_AP_detection_dataset_metadata()

generate_binary_dataset_pdf_report(
    metadata=metadata,
    split_report=split_report,
    error_report=error_report,
    output_paths=[
        (
            settings.ap_detection_config
            / settings.ap_detection_version
            / "config"
            / "passport.pdf"
        ).resolve(),
        (
            Path("reports")
            / "ap_detection_dataset_split_report.pdf"
        ).resolve(),
    ],
)

## Atelectasis Diagnosis

 * Clinical condition: Atelectasis
 * Control group: Healthy
 * patient views: AP, PA
 * Aquisition parameters: D90, D110, S120
 * mAs range: (0,6.9]

## Pleural effusion

 * Clinical condition: Atelectasis
 * Control group: Healthy
 * patient views: AP, PA
 * Aquisition parameters: D90, D110, S120
 * mAs range: (0,6.9]

## Cardiomegality


 * Clinical condition: Atelectasis
 * Control group: Healthy
 * patient views: AP, PA
 * Aquisition parameters: D90, D110, S120
 * mAs range: (0,6.9]

# Validation

In [ ]:
# TBA